# Qwen3 teacher 규모 비교

공개 Cartridges Qwen benchmark 예제의 학습·평가 설정을 사용한다.

| 조건 | 생성 모델 | Scoring teacher | Student |
|---|---|---|---|
| A | Qwen3-4B | Qwen3-4B | Qwen3-4B |
| B | Qwen3-4B | Qwen3-8B | Qwen3-4B |
| C | Qwen3-8B | Qwen3-4B | Qwen3-4B |
| D | Qwen3-8B | Qwen3-8B | Qwen3-4B |

A/B와 C/D는 각각 같은 대화를 사용한다. 주 비교는 B−A다.
LongHealth: 200문항 정확도. MTOB: Kalamang → English 50문장 chrF.

실행 환경: Linux x86_64, Python 3.12, CUDA BF16 GPU. GPU 전체 경로는 검증 전이다.

## 1. 소스

노트북과 같은 Git 저장소 release의 `qwen3-teacher-scaling-source.zip` 다운로드 URL과 `qwen3-teacher-scaling-source.zip.sha256`에 적힌 SHA-256을 입력한다.


In [ ]:
from pathlib import Path
import hashlib
import json
import os
import re
import shutil
import subprocess
import sys
import urllib.request
import zipfile
from IPython.display import display

SOURCE_URL = ''  # @param {type:'string'}
SOURCE_SHA256 = ''  # @param {type:'string'}
ROOT = Path('/content/cartridge-teacher-scaling')
ARCHIVE = Path('/content/qwen3-teacher-scaling-source.zip')
if not SOURCE_URL.strip():
    raise ValueError('release의 qwen3-teacher-scaling-source.zip 다운로드 URL을 입력하세요.')
if not re.fullmatch(r'[a-fA-F0-9]{64}', SOURCE_SHA256):
    raise ValueError('release에 제공된 SHA-256을 입력하세요.')
with urllib.request.urlopen(SOURCE_URL, timeout=120) as response:
    source_bytes = response.read()
if hashlib.sha256(source_bytes).hexdigest() != SOURCE_SHA256.lower():
    raise ValueError('Source checksum mismatch')
ARCHIVE.write_bytes(source_bytes)
print('Source SHA-256:', hashlib.sha256(ARCHIVE.read_bytes()).hexdigest())
with zipfile.ZipFile(ARCHIVE) as source:
    for entry in source.infolist():
        target = (ROOT / entry.filename).resolve()
        if not target.is_relative_to(ROOT.resolve()):
            raise ValueError('Invalid archive path')
        if target.is_file() and target.read_bytes() != source.read(entry):
            raise ValueError('기존 소스와 다른 archive입니다. 새 런타임에서 실행하세요.')
    source.extractall(ROOT)


## 2. 환경

Tokasaurus의 PyTorch 2.6.0·Transformers 4.53.0을 공통 환경으로 사용한다. Python 패키지는 requirements.lock의 버전 목록에서 설치한다.

In [ ]:
if sys.version_info[:2] != (3, 12):
    raise RuntimeError('Python 3.12 런타임이 필요합니다.')
PYTHON = ROOT / '.venv/bin/python'
if not PYTHON.exists():
    subprocess.run([sys.executable, '-m', 'venv', str(ROOT / '.venv')], check=True)
subprocess.run([str(PYTHON), str(ROOT / 'scripts/setup.py')], check=True)
subprocess.run([str(PYTHON), str(ROOT / 'tests/check.py')], check=True)


## 3. 실행 설정

`smoke`는 32대화·64-token cache·2 steps의 실행 검사다. `pilot`은 512대화·16 steps를 사용한다.
`main`은 공개 학습 예제의 두 데이터 shard에 대응하는 생성 모델당 131,072대화를 사용한다.
Main 기본 학습 seed는 42·123·2026이다. `SEEDS`를 입력하면 해당 값으로 실행한다.

In [ ]:
BENCHMARK = 'longhealth'  # @param ['longhealth', 'mtob']
PROFILE = 'smoke'  # @param ['smoke', 'pilot', 'main']
RUN_NAME = 'qwen3-v1-smoke-001'  # @param {type:'string'}
SEEDS = ''  # @param {type:'string'}
USE_DRIVE = True  # @param {type:'boolean'}
BASE = json.loads((ROOT / 'configs/experiment.json').read_text())
TRAIN_SEEDS = list(dict.fromkeys(int(x.strip()) for x in SEEDS.split(','))) if SEEDS.strip() else (BASE['seeds_for_main'] if PROFILE == 'main' else [BASE['seed']])
if not re.fullmatch(r'[A-Za-z0-9_-]+', RUN_NAME):
    raise ValueError('실행 이름에는 영문·숫자·밑줄·하이픈을 사용하세요.')
RUN = Path('/content/runs') / RUN_NAME / BENCHMARK
BACKUP = None
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    BACKUP = Path('/content/drive/MyDrive/cartridge-teacher-scaling') / RUN_NAME / BENCHMARK
    if BACKUP.exists() and not RUN.exists():
        shutil.copytree(BACKUP, RUN)
RUN.mkdir(parents=True, exist_ok=True)
CODE_SHA = hashlib.sha256((ROOT / 'experiment.py').read_bytes()).hexdigest()
PATCH_SHA = hashlib.sha256((ROOT / 'patches/cartridges.patch').read_bytes()).hexdigest()
LOCK_SHA = hashlib.sha256((ROOT / 'requirements.lock').read_bytes()).hexdigest()
STUDY = {'lock_sha256': LOCK_SHA, 'benchmark': BENCHMARK, 'profile': PROFILE, 'config': BASE, 'code_sha256': CODE_SHA, 'patch_sha256': PATCH_SHA}
if (RUN / 'study.json').exists() and json.loads((RUN / 'study.json').read_text()) != STUDY:
    raise ValueError('기존 실행과 설정이 다릅니다. 새 RUN_NAME을 사용하세요.')
(RUN / 'study.json').write_text(json.dumps(STUDY, indent=2))
print(RUN, TRAIN_SEEDS)


In [ ]:
def run_stage(stage, *, generator='4B', teacher='4B', condition='A', seed=42):
    if stage == 'prepare':
        summary = RUN / 'prepare.summary.json'
    elif stage == 'synthesize':
        summary = RUN / f'G{generator}/synthesize.summary.json'
    elif stage == 'score':
        summary = RUN / f'G{generator}/score-{teacher}.summary.json'
    else:
        summary = RUN / f'seed-{seed}/{stage}-{condition}.summary.json'
    if summary.exists():
        result = json.loads(summary.read_text())
        if result['code_sha256'] != CODE_SHA or result['patch_sha256'] != PATCH_SHA or result.get('lock_sha256') != LOCK_SHA:
            raise ValueError('기존 결과와 코드가 다릅니다.')
        return result
    summary.parent.mkdir(parents=True, exist_ok=True)
    command = [str(PYTHON), '-u', str(ROOT / 'experiment.py'), stage,
               '--benchmark', BENCHMARK, '--profile', PROFILE, '--out', str(RUN),
               '--generator', generator, '--teacher', teacher, '--condition', condition, '--seed', str(seed)]
    with summary.with_suffix('.log').open('w') as log:
        process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, cwd=ROOT)
        for line in process.stdout:
            log.write(line); log.flush(); print(line, end='')
        status = process.wait()
    if BACKUP:
        # Model snapshots are reproducible downloads, not experiment outputs.
        shutil.copytree(RUN, BACKUP, dirs_exist_ok=True, ignore=shutil.ignore_patterns('models'))
    if status:
        raise RuntimeError(f'{stage} 실패: {summary.with_suffix(".log")}')
    return json.loads(summary.read_text())


## 4. 데이터

공개 resource sampler에서 batch당 발췌 하나와 32개 seed prompt를 준비한다.

In [ ]:
run_stage('prepare')

## 5. 합성과 scoring

공개 Tokasaurus client와 SelfStudySynthesizer를 사용한다. 서버는 생성 단계가 끝나면 종료된다. 두 teacher가 저장된 답변의 분포를 계산한다.

In [ ]:
for generator in ['4B', '8B']:
    run_stage('synthesize', generator=generator)
    for teacher in ['4B', '8B']:
        display(run_stage('score', generator=generator, teacher=teacher))

## 6. 학습과 평가

공개 TrainConfig의 초기화·loss·패킹·optimizer·주기적 평가를 사용한다. 결과와 checkpoint는 조건·seed별로 저장한다.

In [ ]:
rows = []
for seed in TRAIN_SEEDS:
    for condition in ['A', 'B', 'C', 'D']:
        result = run_stage('train', condition=condition, seed=seed)
        rows.append({'condition': condition, 'seed': seed, **result['evaluations'][-1]})
import pandas as pd
display(pd.DataFrame(rows))

## 실행 결과

결과는 `/content/runs/<RUN_NAME>/<BENCHMARK>`에 저장된다. `USE_DRIVE=True`이면 Google Drive의 `cartridge-teacher-scaling/<RUN_NAME>/<BENCHMARK>`에도 복사된다.

실행 폴더에는 config·환경 정보, 생성 대화·soft targets, 초기 cache, checkpoint·prediction·metric·평가 RNG 상태가 저장된다. Prediction에는 평가 질문과 정답이 포함되며 해당 데이터의 라이선스가 적용된다.
